In [ ]:
import sqlite3 
import pandas as pd 
import numpy as np
import datetime
import plotly.express as px
import plotly
import os
import sys
import ast
from sklearn.cluster import KMeans

In [ ]:
%load_ext autoreload
%autoreload 2

## Read from Database

In [ ]:
look_back_days = 14

In [ ]:
today = datetime.datetime.now() - datetime.timedelta(days=1)
# today = datetime.datetime(2020, 9, 26)

today_str = str(today.date())
one_week_ago = (today - datetime.timedelta(days=look_back_days))
one_week_ago_str = str(one_week_ago.date())
print("Begining on {0} ending on {1}".format(today_str, one_week_ago_str))

In [ ]:
db_name = '/home/malcolm/Spotify_Logger/data/listening_history.db'
con = sqlite3.connect(db_name)
cursor = con.cursor()

In [ ]:
cursor.execute('SELECT name FROM sqlite_master WHERE type=\'table\' ORDER BY name')
tables = cursor.fetchall()
tables = [x[0] for x in tables]
tables

In [ ]:
one_week_query = """select * from Listening_History where played_at_date between '{0}' and '{1}'"""\
    .format(one_week_ago_str, today_str)
one_week_df = pd.read_sql(one_week_query, con)
one_week_df['played_at_date'] = pd.to_datetime(one_week_df['played_at_date'])
# one_week_df['played_at_time'] =  pd.to_datetime(one_week_df['played_at_time'],format= '%H:%M' ).dt.time

In [ ]:
one_week_df.sort_values(['played_at_date', 'played_at_time'], ascending=False).head()

In [ ]:
one_week_df.shape

## Weekly Analysis

In [ ]:
def get_date_metrics(df):
    out = {}
    
    out['# of Songs'] = df.shape[0]
    out['# of Unique Songs'] = df['name'].nunique()
    out['Minutes Played'] = np.round(df['duration_min'].sum(),2)
    out['Hours Played'] = np.round(df['duration_min'].sum()/60,2)
    
    out['Spotify Cost'] = ((today-one_week_ago).days/30) * 9.99
    out["Cost/Hr"] = np.round(out['Spotify Cost']/out['Hours Played'] , 2)
    
    artist_size = df.groupby('artist_name').size()
    most_list_art = artist_size.idxmax()
    song_size = df.groupby('name').size()
    most_list_song = song_size.idxmax()
    
    out['Most Listened Artist'] = most_list_art
    out['Most Listened Song'] = most_list_song
    
    
    out_series = pd.Series(out)
    return(out_series)

In [ ]:
metrics = get_date_metrics(one_week_df)
metrics

In [ ]:
songs_by_date = one_week_df.groupby('played_at_date').apply(get_date_metrics).sort_index(ascending=False)
songs_by_date

In [ ]:
metrics2 = metrics.copy()
metrics2['Date'] = str(datetime.datetime.now().date())
metrics2 = pd.DataFrame(metrics2).T


In [ ]:
metrics2.to_sql('Metrics_WoW', con, index=False, if_exists='append')

In [ ]:
songs_by_date.reset_index().to_sql('Songs_by_date_WoW', con, index=False, if_exists='append')

## Time of Day Listened 

In [ ]:
one_week_df['time_split'] = one_week_df['played_at_time'].str.split(':')
one_week_df['Time Played'] = pd.to_datetime(one_week_df['played_at_time'])
one_week_df['hr'] = one_week_df['time_split'].apply(lambda x: int(x[0]))
# To make night listening continous 
one_week_df['hr_adj'] = one_week_df['hr'].apply( lambda x: 23 + x if x <= 4 else x) 

one_week_df['mins'] = one_week_df['time_split'].apply(lambda x: int(x[1]))

one_week_df['flat_time_modeling'] = 60 * one_week_df['hr_adj'] + one_week_df['mins']  
one_week_df['flat_time'] = 60 * one_week_df['hr'] + one_week_df['mins']  


In [ ]:
kmeans = KMeans(n_clusters=4, random_state=1234)
kmeans.fit(one_week_df['flat_time_modeling'].values.reshape(-1, 1))

In [ ]:
one_week_df['time_grp'] = kmeans.predict(one_week_df['flat_time_modeling'].values.reshape(-1, 1))


In [ ]:
date_diff = (one_week_df['played_at_date'].max() - one_week_df['played_at_date'].min()).days
date_diff

In [ ]:
def get_min_max_time_grp(df):
    out = {}
    out['Count'] = df.shape[0]
    
    min_raw = df['flat_time_modeling'].min()
    min_fin = df[df['flat_time_modeling'] == min_raw]['played_at_time'].iloc[0]
    min_time = df[df['flat_time_modeling'] == min_raw]['Time Played'].iloc[0]
    
    max_raw = df['flat_time_modeling'].max()
    max_fin = df[df['flat_time_modeling'] == max_raw]['played_at_time'].iloc[0]
    max_time = df[df['flat_time_modeling'] == max_raw]['Time Played'].iloc[0]
    
    diff_hrs = (max_time - min_time).seconds/3600
    
    time_range = min_fin + ' - ' + max_fin
    out['Time Range'] = time_range
    out['# of Hours'] = np.round(diff_hrs, 3)
    out[f'Songs/Hr / {date_diff}days'] = np.round(out['Count']/(out['# of Hours'] * date_diff), 3)
#     out[f'Songs/Hr'] = np.round(out['Count']/(out['# of Hours']), 3)    
    
    out_series = pd.Series(out)
    return(out_series)

In [ ]:
listen_grp_stats = one_week_df.groupby('time_grp')\
    .apply(lambda x: get_min_max_time_grp(x))\
    .sort_values('Time Range')
sum_total = listen_grp_stats[f'Songs/Hr / {date_diff}days'].sum()
listen_grp_stats['Pct of Day'] = np.round(100*listen_grp_stats['# of Hours']/listen_grp_stats['# of Hours'].sum()
                                          , 2)
listen_grp_stats['Pct of Songs'] = np.round(100*listen_grp_stats['Count']/listen_grp_stats['Count'].sum(), 2)
listen_grp_stats

In [ ]:
last_line = pd.Series({
    'Count': listen_grp_stats['Count'].sum()
, 'Time Range': np.nan
, '# of Hours': listen_grp_stats['# of Hours'].sum()
, f'Songs/Hr / {date_diff}days': listen_grp_stats['Count'].sum()/(listen_grp_stats['# of Hours'].sum() * date_diff)
, 'Pct of Day': 100.0/listen_grp_stats.shape[0]
, 'Pct of Songs': 100.0/listen_grp_stats.shape[0]
 , 
}, name='Benchmark')
listen_grp_stats = listen_grp_stats.append(last_line)

In [ ]:
grp_mapping = listen_grp_stats['Time Range'].to_dict()
one_week_df['Time Range'] = one_week_df['time_grp'].apply(lambda x: grp_mapping.get(x))

In [ ]:
px.histogram(one_week_df , 'flat_time_modeling', color='Time Range')

In [ ]:
px.histogram(one_week_df , 'Time Played', color='Time Range')

## By Song 

In [ ]:
one_week_df.head()

In [ ]:
def get_song_analysis(df):
    out = {}
    out['Artist'] = df['artist_name'].max()
    out['Length'] = np.round(df['duration_min'].max(), 3)
    out['Spotify Popularity'] = df['popularity'].max()
    out['Times Listened'] = df.shape[0]
    out['Mins Listened this week'] = np.round(df['duration_min'].sum(), 3)
    
    # Group by Date
    date_cnts = df.groupby('played_at_date').count()['artist_name'].sort_values(ascending=False)
    most_listened_to_date = date_cnts.index[0].day_name() + ' ' + str(date_cnts.index[0].strftime('%m-%d'))
    most_listened_to_cnts = date_cnts.iloc[0]
    out['Listened to Most on'] = most_listened_to_date
    out['Cnt of Most Listened Day'] = most_listened_to_cnts
    
    out_series = pd.Series(out)
    return(out_series)

In [ ]:
song_analysis = one_week_df.groupby('name').apply(lambda x:get_song_analysis(x))\
    .sort_values('Times Listened', ascending=False)

In [ ]:
song_analysis_sm = song_analysis[song_analysis['Times Listened'] > 2].head(10)
song_analysis_sm

## Artist Genres

In [ ]:
db_name = 'listening_history.db'
con = sqlite3.connect(f'data/{db_name}')
cursor = con.cursor()

In [ ]:
artist_one_week_query = """select * from Listening_History a
        inner join artists_info b on a.artist_id = b.artist_id 
        where played_at_date between '{0}' and '{1}'"""\
    .format(one_week_ago_str, today_str)
artist_one_week_query.replace('\n', '').replace('  ', '')

In [ ]:
artist_one_week_df = pd.read_sql( artist_one_week_query , con)
artist_one_week_df.head()

In [ ]:
no_genre_stats = {}
no_genre = artist_one_week_df[artist_one_week_df['genres'] == '[]']
no_genre_stats['# of Songs'] = no_genre.shape[0]
no_genre_stats['Mins Listened'] = np.round(no_genre['duration_min'].sum(), 2)
no_genre_stats['% of Songs'] = np.round(100 * no_genre_stats['# of Songs']/ artist_one_week_df.shape[0], 2)
no_genre_stats['% of Mins Listened'] = np.round(100* no_genre_stats['Mins Listened']/
                                               artist_one_week_df['duration_min'].sum(), 2)

no_genre_series = pd.Series(no_genre_stats, name= 'No Genre')
no_genre_series

In [ ]:
genres = {}
artist_one_week_df['genres_list'] = artist_one_week_df['genres'].apply(ast.literal_eval)
for row in artist_one_week_df[['duration_min', 'genres_list']].iterrows():
    genres_list = row[1][1]
    if genres_list == []:
        pass
    else:
        for x in genres_list:
            if x in genres.keys():
                # Increment count, and mins sum if genre not blank 
                genres[x]['# of Songs'] +=1
                genres[x]['Mins Listened'] += row[1][0]
            else:
                # Initialize Genre Dict if genre is new (if not present )
                genres[x] = {'# of Songs': 1, 'Mins Listened': row[1][0]}
    

In [ ]:
genre_dict = pd.DataFrame(genres).T
genre_dict['% of Songs'] = np.round(100 * genre_dict['# of Songs']/artist_one_week_df.shape[0], 2)
genre_dict['% of Mins Listened'] = np.round(100 * genre_dict['Mins Listened']\
                                            /artist_one_week_df['duration_min'].sum(), 2)
genre_dict = genre_dict.sort_values('Mins Listened', ascending=False)
genre_sm = genre_dict.head(10)
genre_sm = genre_sm.append(no_genre_series)
genre_sm

## Genre bump plot

In [ ]:
sys.path.append('/home/malcolm/Spotify_Logger/Slope/')

In [ ]:
from plotSlope import slope


In [ ]:
five_week = today - datetime.timedelta(days=37)
five_week_str = str(five_week.date())
five_week_str

In [ ]:
db_name = 'listening_history.db'
con = sqlite3.connect(f'data/{db_name}')
cursor = con.cursor()

In [ ]:
artist_one_week_query = """select * from Listening_History a
        inner join artists_info b on a.artist_id = b.artist_id 
        where played_at_date between '{0}' and '{1}'"""\
    .format(five_week_str, today_str)
artist_one_week_query.replace('\n', '').replace('  ', '')

In [ ]:
artist_one_week_df = pd.read_sql( artist_one_week_query , con)
artist_one_week_df['played_at_date'] = pd.to_datetime(artist_one_week_df['played_at_date'])
print("Shape: ", artist_one_week_df.shape)
artist_one_week_df.head()

In [ ]:
no_genre_stats = {}
no_genre = artist_one_week_df[artist_one_week_df['genres'] == '[]']
no_genre_stats['# of Songs'] = no_genre.shape[0]
no_genre_stats['Mins Listened'] = np.round(no_genre['duration_min'].sum(), 2)
no_genre_stats['% of Songs'] = np.round(100 * no_genre_stats['# of Songs']/ artist_one_week_df.shape[0], 2)
no_genre_stats['% of Mins Listened'] = np.round(100* no_genre_stats['Mins Listened']/
                                               artist_one_week_df['duration_min'].sum(), 2)

no_genre_series = pd.Series(no_genre_stats, name= 'No Genre')
no_genre_series

In [ ]:
genres = {}
artist_one_week_df['genres_list'] = artist_one_week_df['genres'].apply(ast.literal_eval)
for row in artist_one_week_df[['duration_min', 'genres_list']].iterrows():
    genres_list = row[1][1]
    if genres_list == []:
        pass
    else:
        for x in genres_list:
            if x in genres.keys():
                # Increment count, and mins sum if genre not blank 
                genres[x]['# of Songs'] +=1
                genres[x]['Mins Listened'] += row[1][0]
            else:
                # Initialize Genre Dict if genre is new (if not present )
                genres[x] = {'# of Songs': 1, 'Mins Listened': row[1][0]}
    

In [ ]:
genre_dict = pd.DataFrame(genres).T
genre_dict['% of Songs'] = np.round(100 * genre_dict['# of Songs']/artist_one_week_df.shape[0], 2)
genre_dict['% of Mins Listened'] = np.round(100 * genre_dict['Mins Listened']\
                                            /artist_one_week_df['duration_min'].sum(), 2)
genre_dict = genre_dict.sort_values('Mins Listened', ascending=False)
genre_sm1 = genre_dict.head(10)
genre_sm1 = genre_sm1.append(no_genre_series)
genre_sm1

In [ ]:
top_genres = genre_dict.sort_values('# of Songs', ascending=False).head(7).index.tolist()

In [ ]:
artist_one_week_df_cp = artist_one_week_df.copy()

In [ ]:
for genre in top_genres:
    artist_one_week_df_cp.loc[:, genre] = artist_one_week_df_cp.genres.str.contains(genre)
artist_one_week_df_cp.head()

In [ ]:
corrs = artist_one_week_df_cp[sorted(top_genres)].corr()

so = corrs\
    .unstack()\
    .sort_values(kind="quicksort")\
    .reset_index()\
    .rename({'level_0':'genre1', 'level_1':'genre2', 0:'corr'}, axis=1)
so['genre a1'] = so[['genre1', 'genre2']].apply(lambda x: sorted(x)[0], axis=1)
so['genre a2'] = so[['genre1', 'genre2']].apply(lambda x: sorted(x)[1], axis=1)
so = so\
    .drop(['genre1', 'genre2'], axis=1)\
    .drop_duplicates()
so = so[so['corr'] != 1]
so = so[(so['corr'] < -0.1) | 
       (so['corr'] > 0.2)]
so = so[['genre a1', 'genre a2', 'corr']]
correlations = so.copy()
correlations


In [ ]:
def bump_gb_sum(df, field = 'duration_min', cols = top_genres):
    out = {}
    for x in cols:
        out[x] = df[x].sum()
    return(pd.Series(out))

In [ ]:
dow_today = (today-datetime.timedelta(days=1)).strftime('%a').upper()

In [ ]:
bump_gb1 = artist_one_week_df_cp\
    .set_index('played_at_date')\
    .groupby(pd.Grouper(freq=f'W-{dow_today}'))\
    .apply(lambda x: bump_gb_sum(x))
bump_gb1 = bump_gb1.T
bump_gb1 = bump_gb1.applymap(int).astype(int)
bump_gb1.columns = [str(x.date()) for x in bump_gb1.columns]
bump_gb1

In [ ]:
color = {'latin':'red', 
        'bachata':'orange',
         'pop punk':'blue', 
         'rap':'green', 
        }

In [ ]:
bump_plot_path = f'images/bump_plot_{today_str}.png'

In [ ]:
f = slope(bump_gb1, kind='interval', color = color, font_family='DejaVu Sans',
          height= 18,width=30,font_size=30,dpi=150,
          title = u'Genre Mins Listened During week', savename=bump_plot_path) 

## Playlist Info 

In [ ]:
import math


def millify(n):
    
    millnames = ['',' K',' M',' B',' T']
    n = float(n)
    millidx = max(0,min(len(millnames)-1,
                        int(math.floor(0 if n == 0 else math.log10(abs(n))/3))))

    return '{:.3f}{}'.format(n / 10**(3 * millidx), millnames[millidx])

In [ ]:
def parse_artist_details(spotify, artist_id):
    artist_call = spotify.artist(artist_id)
    out = {}
    out['Name'] = artist_call['name']
    out['Num Tracks'] = 'Followers: ' + millify(artist_call['followers']['total'])
    out['Time (mins)'] = 'Popularity: ' + str(artist_call['popularity'])
    out['First Song'] = str(artist_call['genres'])[1:-1]
    out['Last Song'] = None
    out['Type'] = 'artist'
    return(pd.Series(out))

In [ ]:
def parse_album_details(spotify, album_id):
    album_call = spotify.album(album_id)
    out = {}
    num_artists = len(album_call['artists'])
    artist_part = album_call['artists'][0]['name'] if num_artists == 1\
        else album_call['artists'][0]['name'] + f' - {num_artists} artists' 
    out['Name'] = album_call['name'] + ' - '+ artist_part
    out['Num Tracks'] = album_call['total_tracks']
    out['Time (mins)'] = sum([x['duration_ms'] for x in album_call['tracks']['items']])/60000
    out['First Song'] = album_call['tracks']['items'][0]['name']
    out['Last Song'] = album_call['tracks']['items'][-1]['name']
    out['Type'] = 'album'
    return(pd.Series(out))


In [ ]:

def parse_playlist_details(spotify, playlist_id):
    """
    Safely parse playlist details from the Spotify API.
    Returns a pandas Series with playlist info, or an empty Series if an error occurs.
    """
    expected_cols = ['Name', 'Num Tracks', 'Time (mins)', 'First Song', 'Last Song', 'Type']

    try:
        # Try fetching playlist details
        a_pl = spotify.user_playlist(user='malchemist02', playlist_id=playlist_id)

        out = {}
        out['Name'] = a_pl['name']
        out['Num Tracks'] = a_pl['tracks']['total']
        out['Time (mins)'] = sum(
            [x['track']['duration_ms'] for x in a_pl['tracks']['items']]
        ) / 60000

        # Extract first and last songs
        songs_added = pd.DataFrame(
            [
                [x['track']['name'] + ' ' + x['added_at'][:10], x.get('added_at')]
                for x in a_pl['tracks']['items']
            ],
            columns=['Track Name w Date', 'Date Time Added'],
        )

        first_last_songs = songs_added[
            (songs_added['Date Time Added'] == songs_added['Date Time Added'].min())
            | (songs_added['Date Time Added'] == songs_added['Date Time Added'].max())
        ].sort_values('Date Time Added')

        out['First Song'] = first_last_songs.iloc[0]['Track Name w Date']
        out['Last Song'] = first_last_songs.iloc[-1]['Track Name w Date']
        out['Type'] = 'playlist'

        return pd.Series(out)

    except Exception as e:
        # Print an informative message and return an empty Series with proper columns
        print(f"[Error] Skipping playlist ID '{playlist_id}': {e}")
        empty_out = pd.Series({col: None for col in expected_cols})
        return empty_out

In [ ]:
def parse_source_uri(spotify, uri):
    split_uri = uri.split(':')
    if split_uri[1] == 'playlist':
        details = parse_playlist_details(spotify, split_uri[2])
    elif split_uri[1] == 'album':
        details = parse_album_details(spotify, split_uri[2])
    elif split_uri[1] == 'artist':
        details = parse_artist_details(spotify, split_uri[2])
    else:
        fields = ['Name', 'Num Tracks', 'Time (mins)', 'First Song',
       'Last Song', 'Type']
        details = pd.Series({x : None for x in fields})
    details['uri'] = uri
    return(details)

In [ ]:
from dotenv import load_dotenv
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials, SpotifyOAuth

In [ ]:
load_dotenv()
scope = "user-read-recently-played"
spotify = spotipy.Spotify(client_credentials_manager= SpotifyOAuth(scope=scope
                                                                   , username='malchemist02'))

In [ ]:
playlist_hist = pd.read_sql(f"""
select playlist_id, count(*) as `# of Songs Listened`
, count(distinct  song_uri) as `Unique Songs Listened`
, round(sum(duration_min), 1) as `Mins Listened`
, round(avg(popularity), 1) as `Avg Popularity Listened`
from Listening_History 
where played_at_date > '{one_week_ago_str}'
group by playlist_id
order by 4 desc
""", con)
print("Shape: ", playlist_hist.shape)
playlist_hist.head(10)


In [ ]:
playlist_hist

In [ ]:
source_details = pd.Series(playlist_hist['playlist_id'].dropna())\
    .apply(lambda x:parse_source_uri(spotify, x))
all_playlist_info = pd.merge(playlist_hist, source_details
                             , left_on='playlist_id', right_on='uri', how='outer')
all_playlist_info = all_playlist_info.sort_values('Mins Listened', ascending=False)
all_playlist_info

In [ ]:
playlist_stats1 = all_playlist_info\
    [['Name', 'Unique Songs Listened', 'Mins Listened', '# of Songs Listened', 'Avg Popularity Listened']]\
    .head(7)
playlist_stats2 = all_playlist_info.dropna()\
    [['Name', 'Num Tracks', 'Time (mins)', 'First Song', 'Last Song','Type']].head(7)

In [ ]:
playlist_stats1

In [ ]:
playlist_stats2

## Create HTML

In [ ]:
intro_html_text = f"Here are your Listening Habits for {today_str} through {one_week_ago_str} <br> <br>"

In [ ]:
weekly_stats_html = "<strong> Weekly Highlights </strong> \n \n"
weekly_stats = pd.DataFrame(metrics).rename({0:''}, axis=1)
weekly_stats_html += weekly_stats.to_html()

In [ ]:
by_date_html = "<strong> Analysis by Day </strong> \n\n"
by_date_analysis = songs_by_date[songs_by_date['# of Songs'] >0]
by_date_html = by_date_html + by_date_analysis.to_html()

In [ ]:
song_html = "<strong> Analysis by Song </strong> \n\n"
song_html = song_html + song_analysis_sm.to_html()

In [ ]:
genre_html = "<strong> Top Genres (by Artist Genre) </strong> <br>"
genre_html += """Many songs have multiple genres. Genre's are not exclusive per song.\n 
Genres are also determined by the Primary Artist \n \n """
genre_html = genre_html + genre_sm.to_html()


In [ ]:
playlist_html = "<strong> Top Playlists </strong>\n\n"
playlist_html += playlist_stats1.to_html()
playlist_html += '<br>'
playlist_html += playlist_stats2.to_html()

In [ ]:
time_stats_html = "<strong> Time Group Info </strong>"
time_stats_html += listen_grp_stats.to_html()

In [ ]:
genre_correlations_html = "<strong> 5 week genre corrleations </strong>"
genre_correlations_html += correlations.to_html()

In [ ]:
html_text = " <br> <br> ".join([intro_html_text, weekly_stats_html
                                , by_date_html, song_html, genre_html, playlist_html, time_stats_html
                               , genre_correlations_html])

## Plotting

In [ ]:
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

In [ ]:
date_range = pd.date_range(one_week_ago_str, today_str)
songs_by_date = one_week_df.groupby('played_at_date').apply(get_date_metrics)
songs_by_date.index = pd.DatetimeIndex(songs_by_date.index)
# Fill missing dates with 0 
songs_by_date = songs_by_date.reindex(date_range, fill_value=0)
songs_by_date.index.name = 'Date Listened'
songs_by_date = songs_by_date.reset_index()
songs_by_date

In [ ]:
n_songs_plot = go.Bar(x=songs_by_date['Date Listened'], y=songs_by_date['# of Songs']
                                , text=songs_by_date['# of Songs'], textposition='auto'
                               , showlegend=False)
# n_songs_plot.show()

In [ ]:
n_uni_songs_plt = go.Bar(x=songs_by_date['Date Listened'], y = songs_by_date['# of Unique Songs']
                                  , text=songs_by_date['# of Unique Songs'], textposition='auto'
                        , showlegend=False)
# n_uni_songs_plt.show()

In [ ]:
mins_plot = go.Bar(x=songs_by_date['Date Listened'], y=songs_by_date['Minutes Played']
                            , text=songs_by_date['Minutes Played'], textposition='auto'
                            , showlegend=False)
# mins_plot.update_layout(dict(title='Minutes Played Last Week'))
# mins_plot.update_yaxes(title='Minutes')
# # mins_plot.show()

In [ ]:
metrics_list = metrics.reset_index().T.values.tolist()
metrics_list[0].append('Start & End Date')
metrics_list[1].append(f'{today_str}-{one_week_ago_str}')
metrics_plt = go.Table(header=dict(values=['Metric', 'Values'])
    , cells=dict(values=metrics_list))
# go.Figure(metrics_plt).show()

In [ ]:
# px.histogram(one_week_df , 'Time Played', color='time_grp')
time_fig = go.Figure()
time_range_gb = one_week_df.groupby('Time Range')
for name, grp in time_range_gb:
    time_played_temp = grp['Time Played']
    tmp_trace = go.Histogram(x = time_played_temp, name=name,)
    time_fig.add_trace(tmp_trace)
time_fig.update_layout({'barmode':'stack'
                       , 'title':'# of Songs Played During the Day'
                       , 'yaxis':{
                           'title': 'Number of Songs Played'
                       }})


In [ ]:
fig = make_subplots(rows=2, cols=2
                   , row_heights=[0.45, 0.6]
                   , print_grid=True
                   , subplot_titles=("# of Songs Played", 'Minutes Played'
                                     , '# of Unique Songs', '# of Songs Played During the Day' )
                   , specs=[[{"type": "scatter"}, {"type": "scatter"}],
                           [{"type": "scatter"}, {"type": "bar"}]]
                  )
fig.add_trace(n_songs_plot
             , row=1, col=1)
fig.add_trace(mins_plot
             , row=1, col=2)
fig.add_trace(n_uni_songs_plt
             , row=2, col=1)
for hist in time_fig.data:
    fig.add_trace(hist
             , row=2, col=2)
fig.update_layout(dict(height=700, width=900, barmode='stack'
                       , legend={'xanchor':'left', 'x':1
                                 , 'yanchor':'bottom', 'y': 0.3} ))

In [ ]:
import plotly.io as pio
# pio.orca.config.use_xvfb = True

In [ ]:
img_title = f'images/listening_graphs_{today_str}.png'
# plotly.io.orca.config.executable = ('/usr/local/bin/orca')
try:
    fig.write_image(img_title )
except Exception as e:
    print('Did not save image....')
    print(e)
    pass

## Email

In [ ]:
sys.path.append('/home/malcolm/EmailSender1/')

In [ ]:
from EmailSender import EmailSender


In [ ]:
message_params = {}
message_params['Subject'] = f'Weekly Song Metrics from {today_str} and {one_week_ago_str}'
message_params['Body'] = html_text
message_params['Image_paths'] = [img_title, bump_plot_path]

In [ ]:
email_sender = EmailSender(**message_params)
email_sender.execute()

In [ ]:
con.commit()
con.close()